# Week 3 Assignment 

In [1]:
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

conn = sqlite3.connect(':memory:')
cur = conn.cursor()
print("connected to sqlite")


connected to sqlite


## Loading the dataset

In [2]:
df = pd.read_csv('superstore.csv')
df.columns = [c.replace(' ', '_').replace('-', '_') for c in df.columns]

df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

print(f"rows loaded: {len(df)}")
print(f"columns: {list(df.columns)}")


rows loaded: 9994
columns: ['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [ ]:

pd.read_sql("SELECT * FROM superstore_raw LIMIT 5", conn)

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2020-214675,2020-12-05,2020-12-11,Second Class,CG-52508,Nancy Moore,Consumer,United States,Georgia City,...,63090,South,OFF-ENV-7527,Office Supplies,Envelopes,Poly String Tie Envelopes,23.26,6,0.50,2.51
1,2,CA-2020-771590,2020-01-15,2020-01-17,Same Day,CG-76321,Charles White,Consumer,United States,Virginia City,...,36972,East,TEC-COM-3612,Technology,Computers,Lenovo ThinkPad X1,"10,701.92",10,0.30,"2,323.72"
2,3,CA-2022-934674,2022-12-14,2022-12-21,Standard Class,CG-29790,Ashley Jackson,Home Office,United States,New York City,...,74157,East,OFF-ENV-4543,Office Supplies,Envelopes,Window Envelopes,35.33,6,0.00,0.64
3,4,CA-2022-725969,2022-08-03,2022-08-06,Same Day,CG-23740,Matthew Robinson,Consumer,United States,Virginia City,...,48986,East,OFF-BIN-1841,Office Supplies,Binders,Avery Heavy Duty Binder,37.65,6,0.40,10.02
4,5,CA-2022-804758,2022-04-25,2022-04-26,First Class,CG-50558,Margaret Brown,Consumer,United States,Pennsylvania City,...,60916,East,TEC-COM-3612,Technology,Computers,Lenovo ThinkPad X1,"3,037.81",2,0.00,-110.54


## Creating normalized tables

In [4]:
cur.executescript("""
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;

CREATE TABLE customers (
    customer_id   TEXT PRIMARY KEY,
    customer_name TEXT,
    segment       TEXT,
    region        TEXT
);

CREATE TABLE products (
    product_id   TEXT PRIMARY KEY,
    product_name TEXT,
    category     TEXT,
    sub_category TEXT
);

CREATE TABLE orders (
    row_id      INTEGER PRIMARY KEY,
    order_id    TEXT,
    order_date  TEXT,
    ship_date   TEXT,
    ship_mode   TEXT,
    customer_id TEXT,
    product_id  TEXT,
    sales       REAL,
    quantity    INTEGER,
    discount    REAL,
    profit      REAL
);
""")
conn.commit()
print("tables created")


tables created


In [5]:
cur.executescript("""
INSERT OR IGNORE INTO customers
SELECT DISTINCT Customer_ID, Customer_Name, Segment, Region
FROM superstore_raw;

INSERT OR IGNORE INTO products
SELECT DISTINCT Product_ID, Product_Name, Category, Sub_Category
FROM superstore_raw;

INSERT OR IGNORE INTO orders
SELECT Row_ID, Order_ID, Order_Date, Ship_Date, Ship_Mode,
       Customer_ID, Product_ID, Sales, Quantity, Discount, Profit
FROM superstore_raw;
""")
conn.commit()

for t in ['customers', 'products', 'orders']:
    n = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {t}", conn).iloc[0,0]
    print(f"{t}: {n:,} rows")


customers: 438 rows
products: 37 rows
orders: 9,994 rows


## Subqueries

### 1. Orders where sales are above average

In [6]:
query = """
SELECT
    o.order_id,
    c.customer_name,
    c.segment,
    p.category,
    ROUND(o.sales, 2) as sales,
    ROUND((SELECT AVG(sales) FROM orders), 2) as overall_avg
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p  ON o.product_id  = p.product_id
WHERE o.sales > (SELECT AVG(sales) FROM orders)
ORDER BY o.sales DESC
LIMIT 15
"""

result = pd.read_sql(query, conn)
print(f"orders above average: {pd.read_sql('SELECT COUNT(*) as n FROM orders WHERE sales > (SELECT AVG(sales) FROM orders)', conn).iloc[0,0]:,}")
result


orders above average: 3,102


,order_id,customer_name,segment,category,sales,overall_avg
0,CA-2022-170060,Jessica Robinson,Home Office,Technology,"24,770.60","2,124.11"
1,CA-2021-694655,Sandra Brown,Corporate,Technology,"24,131.58","2,124.11"
2,CA-2020-705173,Daniel Williams,Home Office,Technology,"24,048.12","2,124.11"
3,CA-2020-931882,Nancy Wilson,Corporate,Technology,"23,864.92","2,124.11"
4,CA-2022-343345,James Anderson,Consumer,Technology,"23,802.73","2,124.11"
5,CA-2021-351863,Patricia Martin,Home Office,Technology,"23,755.29","2,124.11"
6,CA-2022-440129,Patricia Davis,Home Office,Technology,"22,566.63","2,124.11"
7,CA-2019-334798,Jessica Brown,Consumer,Technology,"21,661.83","2,124.11"
8,CA-2019-187107,John Martinez,Home Office,Technology,"21,556.07","2,124.11"
9,CA-2022-683715,Matthew Williams,Consumer,Technology,"21,528.16","2,124.11"


### 2. Highest-value order line per customer

In [7]:
query = """
SELECT
    c.customer_name,
    c.segment,
    o.order_id,
    ROUND(o.sales, 2) as max_order_sales
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY max_order_sales DESC
LIMIT 15
"""

pd.read_sql(query, conn)


,customer_name,segment,order_id,max_order_sales
0,Jessica Robinson,Home Office,CA-2022-170060,"24,770.60"
1,Sandra Brown,Corporate,CA-2021-694655,"24,131.58"
2,Daniel Williams,Home Office,CA-2020-705173,"24,048.12"
3,Nancy Wilson,Corporate,CA-2020-931882,"23,864.92"
4,James Anderson,Consumer,CA-2022-343345,"23,802.73"
5,Patricia Martin,Home Office,CA-2021-351863,"23,755.29"
6,Patricia Davis,Home Office,CA-2022-440129,"22,566.63"
7,Jessica Brown,Consumer,CA-2019-334798,"21,661.83"
8,John Martinez,Home Office,CA-2019-187107,"21,556.07"
9,Matthew Williams,Consumer,CA-2022-683715,"21,528.16"


## CTEs (Common Table Expressions)

### 3. Total sales per customer

In [8]:
query = """
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        c.segment,
        c.region,
        ROUND(SUM(o.sales), 2)  as total_sales,
        ROUND(SUM(o.profit), 2) as total_profit,
        COUNT(DISTINCT o.order_id) as num_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name, c.segment, c.region
)
SELECT *
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 20
"""

pd.read_sql(query, conn)


,customer_id,customer_name,segment,region,total_sales,total_profit,num_orders
0,CG-53008,Daniel Williams,Home Office,West,"113,980.62","13,665.36",44
1,CG-40281,Christopher Moore,Consumer,East,"112,892.29","16,348.13",29
2,CG-88249,David Wilson,Consumer,West,"108,281.62","12,219.85",38
3,CG-40234,James Moore,Home Office,South,"107,263.24","13,903.30",27
4,CG-60229,Betty Jones,Corporate,South,"98,538.50","13,288.99",22
5,CG-16824,Matthew Wilson,Home Office,Central,"98,382.01","15,941.86",26
6,CG-55656,Robert Anderson,Home Office,East,"98,147.82","5,317.28",31
7,CG-44469,Sandra Thomas,Consumer,Central,"96,543.49","17,598.45",29
8,CG-43839,Daniel Jones,Home Office,Central,"94,316.32","12,271.18",36
9,CG-81359,Christopher Garcia,Corporate,West,"92,643.43","10,586.38",21


### 4. Sales and profit by category

In [9]:
query = """
WITH cat_summary AS (
    SELECT
        p.category,
        p.sub_category,
        ROUND(SUM(o.sales),  2) as total_sales,
        ROUND(SUM(o.profit), 2) as total_profit,
        COUNT(*) as order_lines,
        ROUND(SUM(o.profit) * 100.0 / SUM(o.sales), 2) as margin_pct
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category, p.sub_category
)
SELECT *
FROM cat_summary
ORDER BY total_profit DESC
"""

pd.read_sql(query, conn)


,category,sub_category,total_sales,total_profit,order_lines,margin_pct
0,Technology,Computers,"7,220,475.65","957,032.14",836,13.25
1,Technology,Phones,"3,569,666.04","468,053.34",832,13.11
2,Furniture,Tables,"3,515,800.52","407,838.74",796,11.60
3,Technology,Copiers,"2,019,367.07","261,284.61",833,12.94
4,Furniture,Chairs,"1,933,062.70","248,307.61",801,12.85
5,Furniture,Bookcases,"1,283,353.78","168,438.42",828,13.12
6,Technology,Machines,"933,495.88","125,062.66",856,13.40
7,Furniture,Furnishings,"298,254.59","37,939.37",875,12.72
8,Office Supplies,Storage,"199,195.30","26,826.45",397,13.47
9,Office Supplies,Accessories,"95,562.66","12,043.81",337,12.60


## Window Functions

### 5. ROW_NUMBER - ranking customers by total sales

In [10]:
query = """
WITH cust_totals AS (
    SELECT
        c.customer_name,
        c.segment,
        c.region,
        ROUND(SUM(o.sales), 2) as total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment, c.region
)
SELECT
    ROW_NUMBER() OVER (ORDER BY total_sales DESC) as row_num,
    customer_name,
    segment,
    region,
    total_sales
FROM cust_totals
ORDER BY row_num
LIMIT 20
"""

pd.read_sql(query, conn)


,row_num,customer_name,segment,region,total_sales
0,1,Daniel Williams,Home Office,West,"113,980.62"
1,2,Christopher Moore,Consumer,East,"112,892.29"
2,3,David Wilson,Consumer,West,"108,281.62"
3,4,James Moore,Home Office,South,"107,263.24"
4,5,Betty Jones,Corporate,South,"98,538.50"
5,6,Matthew Wilson,Home Office,Central,"98,382.01"
6,7,Robert Anderson,Home Office,East,"98,147.82"
7,8,Sandra Thomas,Consumer,Central,"96,543.49"
8,9,Daniel Jones,Home Office,Central,"94,316.32"
9,10,Christopher Garcia,Corporate,West,"92,643.43"


### 6. RANK within each segment

In [11]:
query = """
WITH cust_totals AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment,
        ROUND(SUM(o.sales), 2) as total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment
)
SELECT
    segment,
    customer_name,
    total_sales,
    RANK()       OVER (PARTITION BY segment ORDER BY total_sales DESC) as rank_in_segment,
    DENSE_RANK() OVER (PARTITION BY segment ORDER BY total_sales DESC) as dense_rank
FROM cust_totals
ORDER BY segment, rank_in_segment
LIMIT 24
"""

pd.read_sql(query, conn)


,segment,customer_name,total_sales,rank_in_segment,dense_rank
0,Consumer,Christopher Moore,"112,892.29",1,1
1,Consumer,David Wilson,"108,281.62",2,2
2,Consumer,Sandra Thomas,"96,543.49",3,3
3,Consumer,Barbara Garcia,"85,882.92",4,4
4,Consumer,Jessica Johnson,"85,818.43",5,5
5,Consumer,Matthew Williams,"82,330.82",6,6
6,Consumer,Richard Martinez,"77,887.08",7,7
7,Consumer,Susan Wilson,"77,377.45",8,8
8,Consumer,Jessica Brown,"76,694.79",9,9
9,Consumer,Charles White,"75,955.06",10,10


### 7. Running total of sales per customer over time

In [12]:
query = """
WITH order_totals AS (
    SELECT
        o.customer_id,
        c.customer_name,
        o.order_date,
        ROUND(SUM(o.sales), 2) as daily_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name, o.order_date
),
top5 AS (
    SELECT customer_id
    FROM (
        SELECT customer_id, SUM(daily_sales) as ts
        FROM order_totals
        GROUP BY customer_id
        ORDER BY ts DESC
        LIMIT 5
    )
)
SELECT
    customer_name,
    order_date,
    daily_sales,
    ROUND(SUM(daily_sales) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) as running_total
FROM order_totals
WHERE customer_id IN (SELECT customer_id FROM top5)
ORDER BY customer_name, order_date
LIMIT 30
"""

pd.read_sql(query, conn)


,customer_name,order_date,daily_sales,running_total
0,Betty Jones,2019-01-23,"5,399.82","5,399.82"
1,Betty Jones,2019-05-28,"3,266.80","8,666.62"
2,Betty Jones,2019-10-23,"1,519.41","10,186.03"
3,Betty Jones,2019-12-15,108.86,"10,294.89"
4,Betty Jones,2019-12-30,376.85,"10,671.74"
5,Betty Jones,2020-09-16,"4,196.83","14,868.57"
6,Betty Jones,2020-11-05,614.36,"15,482.93"
7,Betty Jones,2020-11-09,"1,670.70","17,153.63"
8,Betty Jones,2021-01-14,"6,240.81","23,394.44"
9,Betty Jones,2021-01-31,530.65,"23,925.09"


## Combining JOIN + CTE + Window Functions

### 8. Final customer summary with rankings

In [13]:
query = """
WITH customer_metrics AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment,
        c.region,
        COUNT(DISTINCT o.order_id)     as total_orders,
        ROUND(SUM(o.sales),  2)        as total_sales,
        ROUND(SUM(o.profit), 2)        as total_profit,
        ROUND(AVG(o.discount) * 100, 1) as avg_discount_pct,
        ROUND(SUM(o.profit) * 100.0 / SUM(o.sales), 2) as profit_margin_pct
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment, c.region
)
SELECT
    RANK() OVER (ORDER BY total_sales DESC)  as sales_rank,
    customer_name,
    segment,
    region,
    total_orders,
    total_sales,
    total_profit,
    avg_discount_pct,
    profit_margin_pct,
    RANK() OVER (ORDER BY total_profit DESC) as profit_rank,
    DENSE_RANK() OVER (PARTITION BY segment ORDER BY total_sales DESC) as rank_in_segment
FROM customer_metrics
ORDER BY sales_rank
LIMIT 25
"""

pd.read_sql(query, conn)


,sales_rank,customer_name,segment,region,total_orders,total_sales,total_profit,avg_discount_pct,profit_margin_pct,profit_rank,rank_in_segment
0,1,Daniel Williams,Home Office,West,44,"113,980.62","13,665.36",18.20,11.99,19,1
1,2,Christopher Moore,Consumer,East,29,"112,892.29","16,348.13",15.50,14.48,2,1
2,3,David Wilson,Consumer,West,38,"108,281.62","12,219.85",17.40,11.29,36,2
3,4,James Moore,Home Office,South,27,"107,263.24","13,903.30",13.00,12.96,17,2
4,5,Betty Jones,Corporate,South,22,"98,538.50","13,288.99",19.60,13.49,22,1
5,6,Matthew Wilson,Home Office,Central,26,"98,382.01","15,941.86",19.60,16.20,4,3
6,7,Robert Anderson,Home Office,East,31,"98,147.82","5,317.28",12.90,5.42,236,4
7,8,Sandra Thomas,Consumer,Central,29,"96,543.49","17,598.45",15.90,18.23,1,3
8,9,Daniel Jones,Home Office,Central,36,"94,316.32","12,271.18",13.60,13.01,35,5
9,10,Christopher Garcia,Corporate,West,21,"92,643.43","10,586.38",13.80,11.43,58,2


## Business Queries

### BQ1 - Top 10 customers by revenue

In [14]:
query = """
WITH cust_rev AS (
    SELECT
        c.customer_name,
        c.segment,
        c.region,
        ROUND(SUM(o.sales),  2) as revenue,
        ROUND(SUM(o.profit), 2) as profit
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id
)
SELECT
    ROW_NUMBER() OVER (ORDER BY revenue DESC) as rank,
    customer_name, segment, region, revenue, profit
FROM cust_rev
ORDER BY revenue DESC
LIMIT 10
"""

pd.read_sql(query, conn)


,rank,customer_name,segment,region,revenue,profit
0,1,Daniel Williams,Home Office,West,"113,980.62","13,665.36"
1,2,Christopher Moore,Consumer,East,"112,892.29","16,348.13"
2,3,David Wilson,Consumer,West,"108,281.62","12,219.85"
3,4,James Moore,Home Office,South,"107,263.24","13,903.30"
4,5,Betty Jones,Corporate,South,"98,538.50","13,288.99"
5,6,Matthew Wilson,Home Office,Central,"98,382.01","15,941.86"
6,7,Robert Anderson,Home Office,East,"98,147.82","5,317.28"
7,8,Sandra Thomas,Consumer,Central,"96,543.49","17,598.45"
8,9,Daniel Jones,Home Office,Central,"94,316.32","12,271.18"
9,10,Christopher Garcia,Corporate,West,"92,643.43","10,586.38"


### BQ2 - Bottom 10 customers (least revenue)

In [15]:
query = """
WITH cust_rev AS (
    SELECT
        c.customer_name,
        c.segment,
        c.region,
        ROUND(SUM(o.sales),  2) as revenue,
        ROUND(SUM(o.profit), 2) as profit,
        COUNT(DISTINCT o.order_id) as num_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id
)
SELECT customer_name, segment, region, revenue, profit, num_orders
FROM cust_rev
ORDER BY revenue ASC
LIMIT 10
"""

pd.read_sql(query, conn)


,customer_name,segment,region,revenue,profit,num_orders
0,Betty Taylor,Home Office,Central,"9,024.17",686.47,17
1,Christopher Martin,Corporate,East,"11,862.47",910.49,11
2,Karen Taylor,Home Office,Central,"13,498.38","1,131.31",11
3,Charles Moore,Consumer,East,"13,587.12",174.43,14
4,Anthony Davis,Home Office,South,"13,770.08","1,255.19",18
5,Jessica Martin,Corporate,South,"15,490.56","2,375.76",23
6,Jessica Jackson,Consumer,West,"16,413.16","3,305.92",21
7,Betty Harris,Corporate,East,"16,796.69","2,334.07",15
8,Michael Harris,Corporate,Central,"17,129.94","1,593.54",18
9,Christopher Robinson,Consumer,West,"17,838.25","2,124.61",14


### BQ3 - Customers with only one order

In [16]:
query = """
WITH order_counts AS (
    SELECT customer_id, COUNT(DISTINCT order_id) as num_orders
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    c.segment,
    c.region,
    ROUND(SUM(o.sales), 2) as total_sales
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.customer_id IN (
    SELECT customer_id FROM order_counts WHERE num_orders = 1
)
GROUP BY c.customer_id
ORDER BY total_sales DESC
"""

result = pd.read_sql(query, conn)
print(f"customers with only 1 order: {len(result)}")
result.head(15)


customers with only 1 order: 0


,customer_name,segment,region,total_sales


### BQ4 - Orders above average sales within their category

In [17]:
query = """
SELECT
    o.order_id,
    c.customer_name,
    p.category,
    p.sub_category,
    ROUND(o.sales, 2) as sales,
    ROUND((
        SELECT AVG(o2.sales)
        FROM orders o2
        JOIN products p2 ON o2.product_id = p2.product_id
        WHERE p2.category = p.category
    ), 2) as category_avg
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
WHERE o.sales > (
    SELECT AVG(o2.sales)
    FROM orders o2
    JOIN products p2 ON o2.product_id = p2.product_id
    WHERE p2.category = p.category
)
ORDER BY p.category, o.sales DESC
LIMIT 20
"""

pd.read_sql(query, conn)


,order_id,customer_name,category,sub_category,sales,category_avg
0,CA-2021-771038,Mark Davis,Furniture,Tables,"14,721.25","2,130.45"
1,CA-2019-292947,Christopher Wilson,Furniture,Tables,"14,635.77","2,130.45"
2,CA-2021-166246,Margaret Miller,Furniture,Tables,"13,240.08","2,130.45"
3,CA-2021-657776,Thomas Harris,Furniture,Tables,"13,183.60","2,130.45"
4,CA-2021-991961,Jessica Jones,Furniture,Tables,"13,012.70","2,130.45"
5,CA-2022-566973,Ashley Thomas,Furniture,Tables,"12,922.39","2,130.45"
6,CA-2022-732608,Betty Jackson,Furniture,Tables,"12,823.35","2,130.45"
7,CA-2022-373295,Daniel Martin,Furniture,Tables,"12,774.53","2,130.45"
8,CA-2020-239458,Sandra Brown,Furniture,Tables,"12,759.03","2,130.45"
9,CA-2019-386525,Michael Robinson,Furniture,Tables,"12,496.27","2,130.45"


### BQ5 - Region-wise sales summary

In [18]:
query = """
WITH region_stats AS (
    SELECT
        c.region,
        ROUND(SUM(o.sales),  2) as total_sales,
        ROUND(SUM(o.profit), 2) as total_profit,
        COUNT(DISTINCT o.order_id)      as total_orders,
        COUNT(DISTINCT o.customer_id)   as unique_customers,
        ROUND(SUM(o.profit) * 100.0 / SUM(o.sales), 2) as margin_pct
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.region
)
SELECT
    RANK() OVER (ORDER BY total_sales DESC) as rank,
    region, total_sales, total_profit, total_orders, unique_customers, margin_pct
FROM region_stats
ORDER BY rank
"""

pd.read_sql(query, conn)


,rank,region,total_sales,total_profit,total_orders,unique_customers,margin_pct
0,1,South,"5,946,415.53","762,774.02",2809,120,12.83
1,2,Central,"5,444,587.17","733,610.87",2525,116,13.47
2,3,East,"5,009,062.97","618,322.14",2333,103,12.34
3,4,West,"4,828,245.30","617,994.98",2249,99,12.80


## Insights

In [19]:
top_cust = pd.read_sql("""
    SELECT c.customer_name, ROUND(SUM(o.sales),2) as ts
    FROM orders o JOIN customers c ON o.customer_id=c.customer_id
    GROUP BY c.customer_id ORDER BY ts DESC LIMIT 1
""", conn)

top_region = pd.read_sql("""
    SELECT c.region, ROUND(SUM(o.sales),2) as ts
    FROM orders o JOIN customers c ON o.customer_id=c.customer_id
    GROUP BY c.region ORDER BY ts DESC LIMIT 1
""", conn)

top_cat = pd.read_sql("""
    SELECT p.category, ROUND(SUM(o.profit),2) as tp
    FROM orders o JOIN products p ON o.product_id=p.product_id
    GROUP BY p.category ORDER BY tp DESC LIMIT 1
""", conn)

print("Key findings from the analysis:")
print()
print(f"- Top customer by revenue: {top_cust.iloc[0]['customer_name']} (${top_cust.iloc[0]['ts']:,.2f})")
print(f"- Highest sales region: {top_region.iloc[0]['region']} (${top_region.iloc[0]['ts']:,.2f})")
print(f"- Most profitable category: {top_cat.iloc[0]['category']} (${top_cat.iloc[0]['tp']:,.2f} profit)")
print()
print("Subqueries were used to filter rows against aggregated values like average sales.")
print("CTEs helped break down multi-step aggregations into readable blocks.")
print("Window functions allowed ranking without losing row-level detail.")


Key findings from the analysis:

- Top customer by revenue: Daniel Williams ($113,980.62)
- Highest sales region: South ($5,946,415.53)
- Most profitable category: Technology ($1,811,432.75 profit)

Subqueries were used to filter rows against aggregated values like average sales.
CTEs helped break down multi-step aggregations into readable blocks.
Window functions allowed ranking without losing row-level detail.


In [20]:
conn.close()
